# 03 · Gold Layer

The gold layer contains the dimensional and fact tables of the final data mart.

**SCD strategies used:**
- **SCD Type 1** (overwrite): `dim_date`, `dim_geography`, `dim_delivery_method` + non-tracked columns of the dimensions below
- **SCD Type 2** (history): specific fields in `dim_product`, `dim_customer`, `dim_employee`
  - `dim_product`: `brand`, `size`, `tax_rate`, `lead_time_days`
  - `dim_customer`: `customer_category_name`, `buying_group_name`, `is_on_credit_hold`
  - `dim_employee`: `is_salesperson`

---
**SCD Type 2 — strategy:**
1. Close records where tracked fields changed (`end_date = today`, `is_active = FALSE`)
2. Update SCD1 columns in-place on records that remained current
3. Insert new versions + new business keys (`start_date = today`, `is_active = TRUE`)

## 1. Imports and connections

Connection parameters are loaded and the data warehouse engine is created so the notebook can write the final gold tables.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, date, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

engine_dwh = create_engine(
    f"postgresql+psycopg2://{DWH_USER}:{DWH_PASSWORD}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f" Engine created.")
print(f"  DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")
print(f"  Start: {run_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")

## 2. SCD Type 1 loading functions

These helper functions manage SCD Type 1 and Type 2 loads in the gold layer, including truncate/reload and bulk upsert semantics.

In [ ]:
from datetime import date as _date

def load_scd1_truncate(df: pd.DataFrame, gold_table: str, schema: str = "gold") -> int:
    """
    Reloads a dimension via TRUNCATE + INSERT.
    Suitable for small dimensions (e.g., dim_delivery_method).
    The surrogate key (SERIAL) is managed by PostgreSQL.
    """
    with engine_dwh.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {schema}.{gold_table} RESTART IDENTITY CASCADE"))
        df.to_sql(gold_table, conn, schema=schema, if_exists="append", index=False)
    return len(df)


def upsert_scd1_bulk(df: pd.DataFrame, gold_table: str, business_key: str,
                     update_cols: list, schema: str = "gold") -> dict:
    """
    Bulk UPSERT via temporary table — much more efficient for large volumes.
    1. Loads df into a temp table
    2. INSERT ... ON CONFLICT DO UPDATE from the temp table
    3. Returns approximate counts
    """
    tmp_table = f"_tmp_{gold_table}"
    col_list  = ", ".join(df.columns.tolist())
    set_clause = ", ".join([f"{c} = EXCLUDED.{c}" for c in update_cols])

    with engine_dwh.begin() as conn:
        # 1. Load into temp
        df.to_sql(tmp_table, conn, schema="public", if_exists="replace", index=False)

        # 2. Count existing (approximation)
        existing = conn.execute(
            text(f"""
                SELECT COUNT(*) FROM public.{tmp_table} t
                WHERE EXISTS (
                    SELECT 1 FROM {schema}.{gold_table} g
                    WHERE g.{business_key} = t.{business_key}
                )
            """)
        ).scalar() or 0

        # 3. UPSERT
        conn.execute(text(f"""
            INSERT INTO {schema}.{gold_table} ({col_list})
            SELECT {col_list} FROM public.{tmp_table}
            ON CONFLICT ({business_key})
            DO UPDATE SET {set_clause}
        """))

        # 4. Clean up temp
        conn.execute(text(f"DROP TABLE IF EXISTS public.{tmp_table}"))

    return {"total": len(df), "updated_approx": existing, "inserted_approx": len(df) - existing}


def upsert_scd2_bulk(df: pd.DataFrame, gold_table: str, business_key: str,
                     scd2_cols: list, scd1_cols: list, schema: str = "gold") -> dict:
    """
    Hybrid SCD Type 2 + Type 1 (bulk via temporary table).
      - scd2_cols: change to close current record, insert new version
      - scd1_cols: change to update in-place on the current record
    Control columns: start_date, end_date, is_active
    """
    tmp   = f"_tmp_{gold_table}"
    today = _date.today()
    far   = _date(9999, 12, 31)

    with engine_dwh.begin() as conn:
        # 1. Source to temp
        df.to_sql(tmp, conn, schema="public", if_exists="replace", index=False)

        # 2. Close current records where any SCD2 field changed
        scd2_changed = " OR ".join([f"g.{c} IS DISTINCT FROM t.{c}" for c in scd2_cols])
        n_closed = conn.execute(text(f"""
            UPDATE {schema}.{gold_table} g
            SET end_date = :today, is_active = FALSE
            FROM public.{tmp} t
            WHERE g.{business_key} = t.{business_key}
              AND g.is_active = TRUE
              AND ({scd2_changed})
        """), {"today": today}).rowcount

        # 3. SCD1: update non-tracked columns on records that remained current
        if scd1_cols:
            scd1_set = ", ".join([f"{c} = t.{c}" for c in scd1_cols])
            n_scd1 = conn.execute(text(f"""
                UPDATE {schema}.{gold_table} g
                SET {scd1_set}
                FROM public.{tmp} t
                WHERE g.{business_key} = t.{business_key}
                  AND g.is_active = TRUE
            """)).rowcount
        else:
            n_scd1 = 0

        # 4. Insert new versions (closed + new business keys)
        col_list = ", ".join(df.columns.tolist())
        n_inserted = conn.execute(text(f"""
            INSERT INTO {schema}.{gold_table} ({col_list}, start_date, end_date, is_active)
            SELECT {col_list}, :today, :far, TRUE
            FROM public.{tmp} t
            WHERE NOT EXISTS (
                SELECT 1 FROM {schema}.{gold_table} g
                WHERE g.{business_key} = t.{business_key}
                  AND g.is_active = TRUE
            )
        """), {"today": today, "far": far}).rowcount

        # 5. Clean up temp
        conn.execute(text(f"DROP TABLE IF EXISTS public.{tmp}"))

    return {
        "total_source":  len(df),
        "new_versions":  n_closed,
        "inserted_new":  n_inserted - n_closed,
        "scd1_updated":  n_scd1,
    }


def get_surrogate_map(gold_table: str, business_key: str, surrogate_key: str,
                      current_only: bool = False) -> dict:
    """Returns {business_key_value: surrogate_key_value} for all dimension records."""
    where = " WHERE is_active = TRUE" if current_only else ""
    sql = f"SELECT {business_key}, {surrogate_key} FROM gold.{gold_table}{where}"
    with engine_dwh.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    return dict(zip(df[business_key], df[surrogate_key]))


print(" SCD Type 1 and Type 2 loading functions defined.")

## 3. `dim_date`

The date dimension is generated from the actual invoice and order date range in the silver layer, then loaded into the gold schema.

In [ ]:
# Determine date range
sql_min_max = """
    SELECT
        LEAST(
            MIN(invoice_date),
            (SELECT MIN(order_date) FROM silver.stg_orders),
            (SELECT MIN(expected_delivery_date) FROM silver.stg_orders)
        ) AS date_min,
        GREATEST(
            MAX(invoice_date),
            (SELECT MAX(order_date) FROM silver.stg_orders),
            (SELECT MAX(expected_delivery_date) FROM silver.stg_orders)
        ) AS date_max
    FROM silver.stg_invoices
"""
with engine_dwh.connect() as conn:
    row = conn.execute(text(sql_min_max)).fetchone()

date_min = row[0] if row and row[0] else date(2013, 1, 1)
date_max = row[1] if row and row[1] else date(2016, 12, 31)

print(f"Date range: {date_min}  to  {date_max}")

# Generate calendar
day_names_en = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
month_names_en = ["", "January", "February", "March", "April", "May", "June",
                  "July", "August", "September", "October", "November", "December"]

date_range = pd.date_range(start=date_min, end=date_max, freq="D")

df_date = pd.DataFrame({"full_date": date_range.date})
df_date["date_key"]     = pd.to_datetime(df_date["full_date"]).dt.strftime("%Y%m%d").astype(int)
df_date["day"]          = pd.to_datetime(df_date["full_date"]).dt.day
df_date["month"]        = pd.to_datetime(df_date["full_date"]).dt.month
df_date["month_name"]   = df_date["month"].map(lambda m: month_names_en[int(m)])  
df_date["quarter"]      = pd.to_datetime(df_date["full_date"]).dt.quarter
df_date["year"]         = pd.to_datetime(df_date["full_date"]).dt.year
df_date["week_of_year"] = pd.to_datetime(df_date["full_date"]).dt.isocalendar().week.astype(int)
df_date["day_of_week"]  = pd.to_datetime(df_date["full_date"]).dt.weekday + 1  # 1=Monday
df_date["day_name"]     = (pd.to_datetime(df_date["full_date"]).dt.weekday
                           .map(lambda d: day_names_en[int(d)]))  
df_date["is_weekend"]   = df_date["day_of_week"] >= 6
df_date["is_holiday"]   = False  # no holiday data in the source

df_date = df_date[[
    "date_key", "full_date", "day", "month", "month_name",
    "quarter", "year", "week_of_year", "day_of_week", "day_name",
    "is_weekend", "is_holiday"
]]

# Insert into gold (TRUNCATE CASCADE + reload)
with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_date CASCADE"))
    df_date.to_sql("dim_date", conn, schema="gold", if_exists="append", index=False)

print(f" gold.dim_date   — {len(df_date)} days ({date_min} to {date_max})")
df_date.head(3)

## 4. `dim_geography`

The geography dimension is built from silver geography data, but only the final fields needed in the gold model are kept.

SCD Type 1: TRUNCATE + RELOAD.

In [ ]:
with engine_dwh.connect() as conn:
    df_geo = pd.read_sql(text("SELECT * FROM silver.stg_geography"), conn)

# Gold dim_geography:
gold_geo = df_geo[["city_name", "state_province_name", "country_name"]].copy()

n = load_scd1_truncate(gold_geo, "dim_geography")
print(f" gold.dim_geography — {n} records")

# Map city_id to geography_key for later lookups
# Join silver (with city_id) with gold (with geography_key) via names
with engine_dwh.connect() as conn:
    df_gold_geo = pd.read_sql(text("SELECT geography_key, city_name, state_province_name, country_name FROM gold.dim_geography"), conn)

geo_lookup = df_geo[["city_id", "city_name", "state_province_name", "country_name"]].merge(
    df_gold_geo, on=["city_name", "state_province_name", "country_name"], how="left"
).drop_duplicates("city_id")

geo_map = dict(zip(geo_lookup["city_id"], geo_lookup["geography_key"]))
print(f"  Geography map: {len(geo_map)} entries")

## 5. `dim_delivery_method`

Delivery methods are loaded as a small SCD Type 1 dimension from the silver staging table (TRUNCATE + reload).

In [ ]:
with engine_dwh.connect() as conn:
    df_dm = pd.read_sql(text("SELECT * FROM silver.stg_delivery_methods"), conn)

gold_dm = df_dm[["delivery_method_id", "delivery_method_name"]].copy()

n = load_scd1_truncate(gold_dm, "dim_delivery_method")
print(f" gold.dim_delivery_method — {n} records")

dm_map = get_surrogate_map("dim_delivery_method", "delivery_method_id", "delivery_method_key")
print(f"  Delivery method map: {len(dm_map)} entries")

## 6. `dim_product`

The product dimension applies SCD Type 2 for business-sensitive attributes and SCD Type 1 for the remaining fields.

- SCD Type 2 on fields: `brand`, `size`, `tax_rate`, `lead_time_days`.  
- SCD Type 1 (overwrite) on the rest: `stock_item_name`, `color_name`, `unit_package_name`, `outer_package_name`, `quantity_per_outer`, `is_chiller_stock`.

In [ ]:
with engine_dwh.connect() as conn:
    df_prod = pd.read_sql(text("SELECT * FROM silver.stg_products"), conn)

gold_prod = df_prod[[
    "stock_item_id", "stock_item_name", "color_name",
    "unit_package_name", "outer_package_name", "brand", "size",
    "lead_time_days", "quantity_per_outer", "is_chiller_stock",
    "tax_rate"
]].copy()

scd2_cols_prod = ["brand", "size", "tax_rate", "lead_time_days"]
scd1_cols_prod = [
    "stock_item_name", "color_name",
    "unit_package_name", "outer_package_name",
    "quantity_per_outer", "is_chiller_stock"
]

result = upsert_scd2_bulk(gold_prod, "dim_product", "stock_item_id",
                          scd2_cols_prod, scd1_cols_prod)

print(f" gold.dim_product — source={result['total_source']}  "
      f"new_versions={result['new_versions']}  "
      f"inserted_new={result['inserted_new']}  "
      f"scd1_updated={result['scd1_updated']}")

prod_map = get_surrogate_map("dim_product", "stock_item_id", "product_key", current_only=True)
print(f"  Product map (current): {len(prod_map)} entries")

## 7. `dim_customer`

The customer dimension uses hybrid SCD logic and resolves geography keys from the silver geography mapping.

- SCD Type 2 on fields: `customer_category_name`, `buying_group_name`, `is_on_credit_hold`.  
- SCD Type 1 on the rest: `customer_name`, `bill_to_customer_name`, `phone_number`, `geography_key`.  
- Resolves `geography_key` via `city_id` to `dim_geography` map.

In [ ]:
with engine_dwh.connect() as conn:
    df_cust = pd.read_sql(text("SELECT * FROM silver.stg_customers"), conn)

# Resolve geography_key
df_cust["geography_key"] = df_cust["city_id"].map(geo_map)

no_geo = df_cust["geography_key"].isna().sum()
if no_geo:
    print(f"{no_geo} customers without mapped city_id to geography_key = NULL")

df_cust["geography_key"] = df_cust["geography_key"].astype("Int64")

gold_cust = df_cust[[
    "customer_id", "customer_name", "customer_category_name",
    "buying_group_name", "bill_to_customer_name",
    "phone_number", "geography_key", "is_on_credit_hold"
]].copy()

scd2_cols_cust = ["customer_category_name", "buying_group_name", "is_on_credit_hold"]
scd1_cols_cust = ["customer_name", "bill_to_customer_name", "phone_number", "geography_key"]

result = upsert_scd2_bulk(gold_cust, "dim_customer", "customer_id",
                          scd2_cols_cust, scd1_cols_cust)

print(f" gold.dim_customer — source={result['total_source']}  "
      f"new_versions={result['new_versions']}  "
      f"inserted_new={result['inserted_new']}  "
      f"scd1_updated={result['scd1_updated']}")

cust_map = get_surrogate_map("dim_customer", "customer_id", "customer_key", current_only=True)
print(f"  Customer map (current): {len(cust_map)} entries")

## 8. `dim_employee`

The employee dimension loads workforce and salesperson data, tracking changes only for the salesperson flag.

- SCD Type 2 on field: `is_salesperson`.  
- SCD Type 1 on the rest: `full_name`, `preferred_name`, `geography_key`.  
- Resolves `geography_key` via `city_id` to `dim_geography` map.

In [ ]:
with engine_dwh.connect() as conn:
    df_emp = pd.read_sql(text("SELECT * FROM silver.stg_employees"), conn)

df_emp["geography_key"] = df_emp["city_id"].map(geo_map)
df_emp["geography_key"] = df_emp["geography_key"].astype("Int64")

no_geo_emp = df_emp["geography_key"].isna().sum()
if no_geo_emp:
    print(f"{no_geo_emp} employees without mapped city_id to geography_key = NULL")

gold_emp = df_emp[[
    "person_id", "full_name", "preferred_name", "is_salesperson", "geography_key"
]].copy()

scd2_cols_emp = ["is_salesperson"]
scd1_cols_emp = ["full_name", "preferred_name", "geography_key"]

result = upsert_scd2_bulk(gold_emp, "dim_employee", "person_id",
                          scd2_cols_emp, scd1_cols_emp)

print(f" gold.dim_employee — source={result['total_source']}  "
      f"new_versions={result['new_versions']}  "
      f"inserted_new={result['inserted_new']}  "
      f"scd1_updated={result['scd1_updated']}")

emp_map = get_surrogate_map("dim_employee", "person_id", "employee_key", current_only=True)
print(f"  Employee map (current): {len(emp_map)} entries")

## 9. `fact_sales`

The sales fact table is built at invoice-line granularity, resolving all required surrogate foreign keys.

Granularity: **one row per InvoiceLine**.

Surrogate key resolution:
- `date_key`                    ← `stg_invoices.invoice_date` to YYYYMMDD format
- `customer_key`                ← `stg_invoices.customer_id` to `dim_customer`
- `product_key`                 ← `stg_invoice_lines.stock_item_id` to `dim_product`
- `salesperson_employee_key`    ← `stg_invoices.salesperson_person_id` to `dim_employee`
- `delivery_method_key`         ← `stg_invoices.delivery_method_id` to `dim_delivery_method`

Measures:
- `quantity`, `unit_price`, `tax_amount`, `line_total_excl_tax`

In [ ]:
with engine_dwh.connect() as conn:
    df_inv = pd.read_sql(text("SELECT * FROM silver.stg_invoices"), conn)
    df_il  = pd.read_sql(text("SELECT * FROM silver.stg_invoice_lines"), conn)

# Join invoice_lines with invoices (to get all FKs)
df_fs = df_il.merge(
    df_inv[["invoice_id", "customer_id", "salesperson_person_id",
            "delivery_method_id", "invoice_date", "order_id"]],
    on="invoice_id", how="left"
)

# Convert date_key
df_fs["date_key"] = pd.to_datetime(df_fs["invoice_date"]).dt.strftime("%Y%m%d").astype(int)

# Resolve surrogate keys via maps (vectorised)
df_fs["customer_key"]              = df_fs["customer_id"].map(cust_map)
df_fs["product_key"]               = df_fs["stock_item_id"].map(prod_map)
df_fs["salesperson_employee_key"]  = df_fs["salesperson_person_id"].map(emp_map)
df_fs["delivery_method_key"]       = df_fs["delivery_method_id"].map(dm_map)

# Validate unresolved keys
for col in ["customer_key", "product_key", "salesperson_employee_key", "delivery_method_key"]:
    nulls = df_fs[col].isna().sum()
    if nulls:
        print(f" fact_sales.{col}: {nulls} unresolved keys")

# Build fact table
fact_sales = df_fs[[
    "customer_key", "date_key", "delivery_method_key", "product_key",
    "salesperson_employee_key",
    "invoice_id", "order_id",
    "quantity", "unit_price", "tax_amount", "line_total_excl_tax"
]].copy()

# Convert surrogate keys to Int64 (supports NULL)
for col in ["customer_key", "product_key", "salesperson_employee_key", "delivery_method_key"]:
    fact_sales[col] = fact_sales[col].astype("Int64")

# Load into gold (TRUNCATE + reload - fact is always reloaded)
with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_sales"))
    fact_sales.to_sql("fact_sales", conn, schema="gold", if_exists="append", index=False)

print(f" gold.fact_sales — {len(fact_sales)} rows")
fact_sales.head(3)

## 10. `fact_orders`

The orders fact table is built at order-line granularity, resolving date and dimension keys for analytics.

Granularity: **one row per OrderLine**.

Surrogate key resolution:
- `order_date_key`              ← `stg_orders.order_date` to YYYYMMDD format
- `expected_delivery_date_key`  ← `stg_orders.expected_delivery_date` to YYYYMMDD format
- `customer_key`                ← `stg_orders.customer_id` to `dim_customer`
- `product_key`                 ← `stg_order_lines.stock_item_id` to `dim_product`
- `salesperson_employee_key`    ← `stg_orders.salesperson_person_id` to `dim_employee`

Measures:
- `ordered_quantity`, `picked_quantity`, `backordered_quantity`

In [ ]:
with engine_dwh.connect() as conn:
    df_ord = pd.read_sql(text("SELECT * FROM silver.stg_orders"), conn)
    df_ol  = pd.read_sql(text("SELECT * FROM silver.stg_order_lines"), conn)

# Join order_lines with orders
df_fo = df_ol.merge(
    df_ord[["order_id", "customer_id", "salesperson_person_id",
            "order_date", "expected_delivery_date"]],
    on="order_id", how="left"
)

# Convert date keys
df_fo["order_date_key"] = pd.to_datetime(df_fo["order_date"]).dt.strftime("%Y%m%d").astype(int)
df_fo["expected_delivery_date_key"] = pd.to_datetime(df_fo["expected_delivery_date"]).dt.strftime("%Y%m%d").astype("Int64")

# Resolve surrogate keys
df_fo["customer_key"]              = df_fo["customer_id"].map(cust_map)
df_fo["product_key"]               = df_fo["stock_item_id"].map(prod_map)
df_fo["salesperson_employee_key"]  = df_fo["salesperson_person_id"].map(emp_map)

# Validate
for col in ["customer_key", "product_key", "salesperson_employee_key"]:
    nulls = df_fo[col].isna().sum()
    if nulls:
        print(f"fact_orders.{col}: {nulls} unresolved keys")

fact_orders = df_fo[[
    "customer_key", "order_date_key", "product_key",
    "expected_delivery_date_key", "salesperson_employee_key",
    "order_id",
    "ordered_quantity", "picked_quantity", "backordered_quantity"
]].copy()

for col in ["customer_key", "product_key", "salesperson_employee_key", "expected_delivery_date_key"]:
    fact_orders[col] = fact_orders[col].astype("Int64")

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_orders"))
    fact_orders.to_sql("fact_orders", conn, schema="gold", if_exists="append", index=False)

print(f" gold.fact_orders — {len(fact_orders)} rows")
fact_orders.head(3)

## 11. Final data mart validation

A final validation step checks record counts and referential integrity across all gold tables.

In [ ]:
gold_tables = [
    "dim_date", "dim_geography", "dim_delivery_method",
    "dim_product", "dim_customer", "dim_employee",
    "fact_sales", "fact_orders",
]

rows = []
with engine_dwh.connect() as conn:
    for t in gold_tables:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM gold.{t}")).scalar()
        rows.append({"table": f"gold.{t}", "records": cnt})

df_summary = pd.DataFrame(rows)
print("Gold row counts:")
print(df_summary.to_string(index=False))

# Check basic referential integrity - fact_sales
print("\nReferential integrity (fact_sales):")
ref_checks_sales = [
    ("customer_key",              "dim_customer",        "customer_key"),
    ("product_key",               "dim_product",         "product_key"),
    ("salesperson_employee_key",  "dim_employee",        "employee_key"),
    ("delivery_method_key",       "dim_delivery_method", "delivery_method_key"),
]
with engine_dwh.connect() as conn:
    for fk, dim, pk in ref_checks_sales:
        orphans = conn.execute(text(f"""
            SELECT COUNT(*) FROM gold.fact_sales fs
            LEFT JOIN gold.{dim} d ON fs.{fk} = d.{pk}
            WHERE fs.{fk} IS NOT NULL AND d.{pk} IS NULL
        """)).scalar()
        status = "" if orphans == 0 else f"{orphans} orphans"
        print(f"  {status}  fact_sales.{fk} to gold.{dim}")

# Check referential integrity - fact_orders
print("\nReferential integrity (fact_orders):")
ref_checks_orders = [
    ("customer_key",                "dim_customer",  "customer_key"),
    ("product_key",                 "dim_product",   "product_key"),
    ("salesperson_employee_key",    "dim_employee",  "employee_key"),
    ("order_date_key",              "dim_date",      "date_key"),
    ("expected_delivery_date_key",  "dim_date",      "date_key"),
]
with engine_dwh.connect() as conn:
    for fk, dim, pk in ref_checks_orders:
        orphans = conn.execute(text(f"""
            SELECT COUNT(*) FROM gold.fact_orders fo
            LEFT JOIN gold.{dim} d ON fo.{fk} = d.{pk}
            WHERE fo.{fk} IS NOT NULL AND d.{pk} IS NULL
        """)).scalar()
        status = "" if orphans == 0 else f" {orphans} orphans"
        print(f"  {status}  fact_orders.{fk} to gold.{dim}")

print("\n Gold completed. ELT pipeline complete!")
print(f"  Total duration: {(datetime.now(timezone.utc).replace(tzinfo=None) - run_at).seconds}s")

## 12. Example queries - analytical smoke tests

Verifies that the data mart responds to typical business questions.

In [ ]:
# Sales by year and quarter
sql = """
    SELECT d.year, d.quarter,
           COUNT(DISTINCT fs.invoice_id)         AS num_invoices,
           SUM(fs.quantity)                      AS total_qty,
           ROUND(SUM(fs.line_total_excl_tax), 2) AS total_revenue
    FROM   gold.fact_sales fs
    JOIN   gold.dim_date   d  ON fs.date_key = d.date_key
    GROUP  BY d.year, d.quarter
    ORDER  BY d.year, d.quarter;
"""
with engine_dwh.connect() as conn:
    df_q1 = pd.read_sql(text(sql), conn)

print("Sales by year/quarter:")
df_q1

In [ ]:
# Top 10 products by revenue
sql = """
    SELECT dp.stock_item_name,
           SUM(fs.quantity)                      AS total_qty,
           ROUND(SUM(fs.line_total_excl_tax), 2) AS total_revenue
    FROM   gold.fact_sales  fs
    JOIN   gold.dim_product dp ON fs.product_key = dp.product_key
    GROUP  BY dp.stock_item_name
    ORDER  BY total_revenue DESC
    LIMIT  10;
"""
with engine_dwh.connect() as conn:
    df_q2 = pd.read_sql(text(sql), conn)

print("Top 10 products by revenue:")
df_q2

In [ ]:
# Top 10 salespersons with most backorders
sql = """
    SELECT de.full_name,
           SUM(fo.backordered_quantity) AS total_backordered
    FROM   gold.fact_orders  fo
    JOIN   gold.dim_employee de ON fo.salesperson_employee_key = de.employee_key
    WHERE  fo.backordered_quantity > 0
    GROUP  BY de.full_name
    ORDER  BY total_backordered DESC
    LIMIT  10;
"""
with engine_dwh.connect() as conn:
    df_q3 = pd.read_sql(text(sql), conn)

print("Top 10 salespersons with most backorders:")
df_q3